In [1]:
# Import things that are needed generically
from langchain.agents import AgentType, Tool, initialize_agent
from langchain_openai import OpenAI
from langchain.chains import RetrievalQA

from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import Tool
import os
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.tools.file_management.write import WriteFileTool
from langchain.vectorstores import FAISS



In [2]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [3]:
import fitz  # PyMuPDF

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    full_text = ""
    for page in doc:
        full_text += page.get_text()
    return full_text



In [5]:
data_path = "/Users/lama/Desktop/agent /80 Excel formulas .pdf"

In [25]:
from langchain_core.documents import Document
text = extract_text_from_pdf(data_path)
documents = [Document(page_content=text)]


In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Use a text splitter to break the document into smaller pieces
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000)
docs = text_splitter.split_documents(documents)

In [27]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# Create embeddings using OpenAI
embeddings = OpenAIEmbeddings()

# Store the embedded chunks in a FAISS vector database
db = FAISS.from_documents(docs , embeddings)

In [28]:
faiss_retriever = db.as_retriever()

In [29]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

In [30]:
ExcelFormulas= RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=faiss_retriever
)

In [80]:
import requests

# Weather function using wttr.in
def get_weather(input_query=None):
    base_url = "https://wttr.in?format=%C+%t+%w"  # Simple weather query format
    response = requests.get(base_url)
    
    if response.status_code == 200:
        return response.text
    else:
        return "Could not retrieve weather data."

In [81]:
tools = [
    Tool(
        name="Excel Formulas",
        func=ExcelFormulas.run,
        description="useful for when you need to answer questions about Excel formulas, functions, and data manipulation. Input should be a fully formed question.",
    ),
    Tool(
        name="Weather",
        func=get_weather,
        description="Fetches simple weather information without requiring location or API key."
    ),
]

In [82]:
agent = initialize_agent(
    tools, 
    llm, 
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,  # Type of agent (adjust if needed)
    verbose=True  # Enable verbose logging to see interactions  # Pass the memory so it can store history
)

In [83]:
agent.invoke("How is the weather today")



> Entering new AgentExecutor chain...
I need to check the current weather conditions.  
Action: Weather  
Action Input: None  
Observation: Sunny +34°C →20km/h
Thought:I now know the final answer  
Final Answer: The weather today is sunny with a temperature of 34°C and wind speed of 20 km/h.

> Finished chain.


{'input': 'How is the weather today',
 'output': 'The weather today is sunny with a temperature of 34°C and wind speed of 20 km/h.'}

In [84]:
agent.invoke("how to average a column?")



> Entering new AgentExecutor chain...
To average a column in Excel, I need to use the AVERAGE function. I should provide the range of the column I want to average. 

Action: Excel Formulas  
Action Input: "How to use the AVERAGE function to average a column in Excel?"

Observation: To use the AVERAGE function to average a column in Excel, you would enter the formula in a cell where you want the average to appear. The syntax for the AVERAGE function is:

```
=AVERAGE(range)
```

For example, to average the values in column A from rows 1 to 5, you would use the following formula:

```
=AVERAGE(A1:A5)
```

This will calculate the average of the numbers in cells A1 through A5.
Thought:I now know how to average a column in Excel using the AVERAGE function. 

Final Answer: To average a column in Excel, use the AVERAGE function with the syntax `=AVERAGE(range)`. For example, to average the values in column A from rows 1 to 5, use `=AVERAGE(A1:A5)`.

> Finished chain.


{'input': 'how to average a column?',
 'output': 'To average a column in Excel, use the AVERAGE function with the syntax `=AVERAGE(range)`. For example, to average the values in column A from rows 1 to 5, use `=AVERAGE(A1:A5)`.'}